In [ ]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from data.datasets import get_dataset
from data.splits import get_train_val_split

dataset_name = "Oxford-IIIT Pet classification"
data_dir = repo_root / "data" / "raw" / "iit_pet"

# Keep samples raw until statistics have been computed on the training split.
trainval_dataset = get_dataset(
    split="trainval", dataset=dataset_name, data_dir=data_dir, download=False
)
raw_test_dataset = get_dataset(
    split="test", dataset=dataset_name, data_dir=data_dir, download=False
)

raw_train_dataset, raw_val_dataset = get_train_val_split(
    trainval_dataset, ratio=0.2, seed=42, labels=trainval_dataset._labels
)

assert set(raw_train_dataset.indices).isdisjoint(raw_val_dataset.indices)
for name, subset in (("Train", raw_train_dataset), ("Validation", raw_val_dataset)):
    counts = Counter(trainval_dataset._labels[i] for i in subset.indices)
    print(
        f"{name}: {len(subset)} images, {len(counts)} classes "
        f"({min(counts.values())}-{max(counts.values())} per class)"
    )
print(f"Test: {len(raw_test_dataset)} images (official held-out set)")
image, label = raw_train_dataset[0]
print(f"Example: image size={image.size}, label={label}")


In [ ]:
from functools import partial
from torch.utils.data import Subset
from data.datasets import calculate_mean_std
from data.dataloaders import get_dataloader
from data.transforms import image_transform

NUM_WORKERS = 4
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# Estimate RGB statistics using raw training images only.
mean, std = calculate_mean_std(raw_train_dataset, size=IMAGE_SIZE)
print("Train-only RGB mean/std:", mean, std)

train_transform = partial(image_transform, mean=mean, std=std, train=True, size=IMAGE_SIZE)
eval_transform = partial(image_transform, mean=mean, std=std, train=False, size=IMAGE_SIZE)

train_dataset = Subset(
    get_dataset("trainval", dataset_name, data_dir, transform=train_transform),
    raw_train_dataset.indices,
)
val_dataset = Subset(
    get_dataset("trainval", dataset_name, data_dir, transform=eval_transform),
    raw_val_dataset.indices,
)
test_dataset = get_dataset("test", dataset_name, data_dir, transform=eval_transform)

train_dataloader = get_dataloader(
    train_dataset, split="train", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_dataloader = get_dataloader(
    val_dataset, split="val", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
test_dataloader = get_dataloader(
    test_dataset, split="test", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

for split_name, loader in (
    ("train", train_dataloader), ("validation", val_dataloader), ("test", test_dataloader)
):
    images, labels = next(iter(loader))
    print(f"{split_name}: images={tuple(images.shape)}, labels={tuple(labels.shape)}")


In [ ]:
from models import get_classification_model

convnext_model = get_classification_model("convnext_tiny", pretrained=True, num_classes=37)
swin_transformer_model = get_classification_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=37)
resnet50_model = get_classification_model("resnet50", pretrained=True, num_classes=37)

# Restore these initial weights before each independent training strategy.
# This prevents the next experiment from inheriting a previously trained head.
import copy
initial_model_states = {
    "convnext_tiny": copy.deepcopy(convnext_model.state_dict()),
    "swin_tiny": copy.deepcopy(swin_transformer_model.state_dict()),
    "resnet50": copy.deepcopy(resnet50_model.state_dict()),
}


# Head-only classifier training


In [ ]:
import torch
from evaluation import train_one_epoch, validate
from utils.io import save_checkpoint, save_json
from visualization import plot_training_history

NUM_EPOCHS = 10
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
CRITERION = torch.nn.CrossEntropyLoss()
LEARNING_RATE = 1e-4

models = {
    "ConvNeXt Tiny": ("convnext_tiny", convnext_model),
    "Swin Transformer Tiny": ("swin_tiny", swin_transformer_model),
    "ResNet-50": ("resnet50", resnet50_model),
}

training_runs = {}
for model_name, (model_id, model) in models.items():
    print(f"Training classification head: {model_name}")
    model.load_state_dict(initial_model_states[model_id])
    model.to(DEVICE)

    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.get_classifier().parameters():
        parameter.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,}/{total:,} ({trainable/total:.1%})")

    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE
    )
    result_dir = repo_root / "results" / "classification" / "classifier_head" / model_id
    history = []
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_dataloader, CRITERION, optimizer, DEVICE, head_only=True
        )
        val_loss, val_acc = validate(model, val_dataloader, CRITERION, DEVICE)
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
        })
        print(
            f"Epoch {epoch}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_accuracy or (
            val_acc == best_val_accuracy and val_loss < best_val_loss
        ):
            best_epoch = epoch
            best_val_accuracy = val_acc
            best_val_loss = val_loss
            save_checkpoint(
                result_dir / "best.pt", model, optimizer, epoch,
                {"loss": val_loss, "accuracy": val_acc},
                metadata={
                    "model_name": model_name, "model_id": model_id,
                    "dataset": dataset_name, "training_strategy": "head_only",
                    "learning_rate": LEARNING_RATE, "image_size": IMAGE_SIZE,
                    "normalization_mean": mean, "normalization_std": std,
                    "split_seed": 42, "trainable_parameters": trainable,
                    "initialization": "pretrained_backbone_fresh_classifier",
                },
            )

        save_json(history, result_dir / "training_history.json")

    training_runs[model_name] = {
        "model_id": model_id, "history": history,
        "best_epoch": best_epoch, "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
    }
    plot_training_history(history, model_name)
    print(f"Best {model_name}: epoch={best_epoch}, val_acc={best_val_accuracy:.4f}")


In [ ]:
from evaluation import evaluate
from utils.io import load_checkpoint, save_json, save_predictions
from visualization import display_metrics, plot_confusion_matrix, plot_AUC_ROC, plot_AUC_PR

test_results = {}
test_predictions = {}
for model_name, (model_id, model) in models.items():
    result_dir = repo_root / "results" / "classification" / "classifier_head" / model_id
    checkpoint = load_checkpoint(result_dir / "best.pt", model, device=DEVICE)
    metrics, predictions = evaluate(
        model, test_dataloader, CRITERION, DEVICE, return_predictions=True
    )
    test_results[model_name] = metrics
    test_predictions[model_name] = predictions
    save_json(
        {
            "model": model_name,
            "checkpoint_epoch": checkpoint["epoch"],
            "best_validation": checkpoint["validation_metrics"],
            "test_metrics": metrics,
        },
        result_dir / "test_metrics.json",
    )
    save_predictions(predictions, result_dir / "test_predictions.npz")

comparison = display_metrics(test_results)
save_json(test_results, repo_root / "results" / "classification" / "classifier_head" / "comparison.json")


In [ ]:
for name, result in test_results.items():
    predictions = test_predictions[name]
    title = f"{name} ({description})"
    plot_confusion_matrix(
        result["confusion_matrix"], title, class_names=test_dataset.classes
    )
    plot_AUC_ROC(predictions["labels"], predictions["probabilities"], title)
    plot_AUC_PR(predictions["labels"], predictions["probabilities"], title)


# fine tune more layers

In [ ]:
BACKBONE_LEARNING_RATE = 1e-5
HEAD_LEARNING_RATE = 1e-4

training_runs = {}
for model_name, (model_id, model) in models.items():
    print(f"Fine-tuning final stage and classifier: {model_name}")
    model.load_state_dict(initial_model_states[model_id])
    model.to(DEVICE)

    for parameter in model.parameters():
        parameter.requires_grad = False

    if model_id == "convnext_tiny":
        modules_to_train = [model.stages[-1], model.head]
    elif model_id == "swin_tiny":
        modules_to_train = [model.layers[-1], model.norm, model.head]
    elif model_id == "resnet50":
        modules_to_train = [model.layer4, model.fc]
    else:
        raise ValueError(f"Unexpected model: {model_id}")

    for module in modules_to_train:
        for parameter in module.parameters():
            parameter.requires_grad = True

    # Higher learning rate for the freshly initialized classification layer.
    head_params = list(model.get_classifier().parameters())
    head_ids = {id(param) for param in head_params}
    backbone_params = [
        param for param in model.parameters()
        if param.requires_grad and id(param) not in head_ids
    ]
    assert backbone_params and head_params, "Final-stage training needs both groups."

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,}/{total:,} ({trainable/total:.1%})")

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LEARNING_RATE},
        {"params": head_params, "lr": HEAD_LEARNING_RATE},
    ])
    result_dir = repo_root / "results" / "classification" / "partial" / model_id
    history = []
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_dataloader, CRITERION, optimizer, DEVICE, head_only=False
        )
        val_loss, val_acc = validate(model, val_dataloader, CRITERION, DEVICE)
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
        })
        print(
            f"Epoch {epoch}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_accuracy or (
            val_acc == best_val_accuracy and val_loss < best_val_loss
        ):
            best_epoch = epoch
            best_val_accuracy = val_acc
            best_val_loss = val_loss
            save_checkpoint(
                result_dir / "best.pt", model, optimizer, epoch,
                {"loss": val_loss, "accuracy": val_acc},
                metadata={
                    "model_name": model_name, "model_id": model_id,
                    "dataset": dataset_name, "training_strategy": "final_stage",
                    "backbone_learning_rate": BACKBONE_LEARNING_RATE,
                    "head_learning_rate": HEAD_LEARNING_RATE,
                    "image_size": IMAGE_SIZE,
                    "normalization_mean": mean, "normalization_std": std,
                    "split_seed": 42, "trainable_parameters": trainable,
                    "initialization": "pretrained_backbone_fresh_classifier",
                },
            )

        save_json(history, result_dir / "training_history.json")

    training_runs[model_name] = {
        "model_id": model_id, "history": history,
        "best_epoch": best_epoch, "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
    }
    plot_training_history(history, f"{model_name} (final stage)")
    print(f"Best {model_name}: epoch={best_epoch}, val_acc={best_val_accuracy:.4f}")


In [ ]:
test_results = {}
test_predictions = {}
for model_name, (model_id, model) in models.items():
    result_dir = repo_root / "results" / "classification" / "partial" / model_id
    checkpoint = load_checkpoint(result_dir / "best.pt", model, device=DEVICE)
    metrics, predictions = evaluate(
        model, test_dataloader, CRITERION, DEVICE, return_predictions=True
    )
    test_results[model_name] = metrics
    test_predictions[model_name] = predictions
    save_json(
        {
            "model": model_name,
            "checkpoint_epoch": checkpoint["epoch"],
            "best_validation": checkpoint["validation_metrics"],
            "test_metrics": metrics,
        },
        result_dir / "test_metrics.json",
    )
    save_predictions(predictions, result_dir / "test_predictions.npz")

comparison = display_metrics(test_results)
save_json(test_results, repo_root / "results" / "classification" / "partial" / "comparison.json")


In [ ]:
for name, result in test_results.items():
    predictions = test_predictions[name]
    title = f"{name} ({description})"
    plot_confusion_matrix(
        result["confusion_matrix"], title, class_names=test_dataset.classes
    )
    plot_AUC_ROC(predictions["labels"], predictions["probabilities"], title)
    plot_AUC_PR(predictions["labels"], predictions["probabilities"], title)


# fine tune full model

In [ ]:
BACKBONE_LEARNING_RATE = 1e-5
HEAD_LEARNING_RATE = 1e-4

training_runs = {}
for model_name, (model_id, model) in models.items():
    print(f"Fine-tuning entire network: {model_name}")
    model.load_state_dict(initial_model_states[model_id])
    model.to(DEVICE)

    # Restore gradients for every layer (the earlier experiments froze many).
    for parameter in model.parameters():
        parameter.requires_grad = True

    head_params = list(model.get_classifier().parameters())
    head_ids = {id(param) for param in head_params}
    backbone_params = [
        param for param in model.parameters() if id(param) not in head_ids
    ]
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,}/{total:,} ({trainable/total:.1%})")

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LEARNING_RATE},
        {"params": head_params, "lr": HEAD_LEARNING_RATE},
    ])
    result_dir = repo_root / "results" / "classification" / "full" / model_id
    history = []
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_dataloader, CRITERION, optimizer, DEVICE, head_only=False
        )
        val_loss, val_acc = validate(model, val_dataloader, CRITERION, DEVICE)
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
        })
        print(
            f"Epoch {epoch}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_accuracy or (
            val_acc == best_val_accuracy and val_loss < best_val_loss
        ):
            best_epoch = epoch
            best_val_accuracy = val_acc
            best_val_loss = val_loss
            save_checkpoint(
                result_dir / "best.pt", model, optimizer, epoch,
                {"loss": val_loss, "accuracy": val_acc},
                metadata={
                    "model_name": model_name, "model_id": model_id,
                    "dataset": dataset_name, "training_strategy": "full",
                    "backbone_learning_rate": BACKBONE_LEARNING_RATE,
                    "head_learning_rate": HEAD_LEARNING_RATE,
                    "image_size": IMAGE_SIZE,
                    "normalization_mean": mean, "normalization_std": std,
                    "split_seed": 42, "trainable_parameters": trainable,
                    "initialization": "pretrained_backbone_fresh_classifier",
                },
            )

        save_json(history, result_dir / "training_history.json")

    training_runs[model_name] = {
        "model_id": model_id, "history": history,
        "best_epoch": best_epoch, "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
    }
    plot_training_history(history, f"{model_name} (full)")
    print(f"Best {model_name}: epoch={best_epoch}, val_acc={best_val_accuracy:.4f}")


In [ ]:
test_results = {}
test_predictions = {}
for model_name, (model_id, model) in models.items():
    result_dir = repo_root / "results" / "classification" / "full" / model_id
    checkpoint = load_checkpoint(result_dir / "best.pt", model, device=DEVICE)
    metrics, predictions = evaluate(
        model, test_dataloader, CRITERION, DEVICE, return_predictions=True
    )
    test_results[model_name] = metrics
    test_predictions[model_name] = predictions
    save_json(
        {
            "model": model_name,
            "checkpoint_epoch": checkpoint["epoch"],
            "best_validation": checkpoint["validation_metrics"],
            "test_metrics": metrics,
        },
        result_dir / "test_metrics.json",
    )
    save_predictions(predictions, result_dir / "test_predictions.npz")

comparison = display_metrics(test_results)
save_json(test_results, repo_root / "results" / "classification" / "full" / "comparison.json")


In [ ]:
for name, result in test_results.items():
    predictions = test_predictions[name]
    title = f"{name} ({description})"
    plot_confusion_matrix(
        result["confusion_matrix"], title, class_names=test_dataset.classes
    )
    plot_AUC_ROC(predictions["labels"], predictions["probabilities"], title)
    plot_AUC_PR(predictions["labels"], predictions["probabilities"], title)


# Comparison across fine-tuning strategies


In [ ]:
# Side-by-side comparison across all three fine-tuning strategies.
# Use validation results to decide whether to investigate more stages;
# do not tune the experiment based on held-out test performance.
import json

all_test_results = {}
for strategy in ("classifier_head", "partial", "full"):
    for model_name, (model_id, _) in models.items():
        path = (
            repo_root / "results" / "classification" / strategy
            / model_id / "test_metrics.json"
        )
        with path.open(encoding="utf-8") as file:
            saved_result = json.load(file)
        all_test_results[f"{strategy} — {model_name}"] = saved_result["test_metrics"]

all_comparison = display_metrics(all_test_results)
save_json(
    all_test_results, repo_root / "results" / "classification" / "comparison.json"
)
